In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# =========================
# 1. Load data
# =========================
df = pd.read_csv("../data/disease10K.csv")

X = df.drop("has_disease", axis=1)
y = df["has_disease"]

print("Class distribution:")
print(y.value_counts())
print("Positive ratio:", y.mean())

# =========================
# 2. Split data
# =========================
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print("\nShapes:")
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

# =========================
# 3. Preprocessing
# =========================
cat_cols = ["physical_activity_level"]
num_cols = [c for c in X.columns if c not in cat_cols]

pre_onehot = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", "passthrough", num_cols)
])

pre_ordinal = ColumnTransformer([
    ("cat", OrdinalEncoder(categories=[["low", "moderate", "high"]]), cat_cols),
    ("num", "passthrough", num_cols)
])

# =========================
# 4. Imbalance ratio
# =========================
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight_value = neg / pos
print("\nscale_pos_weight:", scale_pos_weight_value)

# =========================
# 5. Threshold search helper
# =========================
def find_best_threshold(y_true, y_proba):
    best = None

    for t in np.arange(0.05, 0.96, 0.05):
        y_pred = (y_proba >= t).astype(int)

        rec = recall_score(y_true, y_pred, zero_division=0)
        prec = precision_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        acc = accuracy_score(y_true, y_pred)

        # prioritize recall, then f1, then precision
        score = (rec, f1, prec, acc)

        if best is None or score > best["score"]:
            best = {
                "score": score,
                "threshold": t,
                "recall": rec,
                "precision": prec,
                "f1": f1,
                "accuracy": acc
            }

    return best

# =========================
# 6. Evaluation helper
# =========================
def evaluate_model(name, model):
    print(f"\n==============================")
    print(f"Running: {name}")
    print(f"==============================")

    model.fit(X_train, y_train)

    # validation threshold tuning
    y_val_proba = model.predict_proba(X_val)[:, 1]
    best_thr = find_best_threshold(y_val, y_val_proba)

    print("Best validation threshold:", best_thr["threshold"])
    print("Validation recall:", best_thr["recall"])
    print("Validation precision:", best_thr["precision"])
    print("Validation f1:", best_thr["f1"])

    # final test evaluation
    y_test_proba = model.predict_proba(X_test)[:, 1]
    y_test_pred = (y_test_proba >= best_thr["threshold"]).astype(int)

    test_acc = accuracy_score(y_test, y_test_pred)
    test_prec = precision_score(y_test, y_test_pred, zero_division=0)
    test_rec = recall_score(y_test, y_test_pred, zero_division=0)
    test_f1 = f1_score(y_test, y_test_pred, zero_division=0)
    test_roc = roc_auc_score(y_test, y_test_proba)
    test_pr = average_precision_score(y_test, y_test_proba)
    cm = confusion_matrix(y_test, y_test_pred)

    print("\nTest Accuracy:", test_acc)
    print("Test Precision:", test_prec)
    print("Test Recall:", test_rec)
    print("Test F1:", test_f1)
    print("Test ROC-AUC:", test_roc)
    print("Test PR-AUC:", test_pr)
    print("\nTest Classification Report:")
    print(classification_report(y_test, y_test_pred))
    print("Confusion Matrix:")
    print(cm)

    return {
        "model": name,
        "threshold": best_thr["threshold"],
        "val_recall": best_thr["recall"],
        "val_precision": best_thr["precision"],
        "val_f1": best_thr["f1"],
        "test_accuracy": test_acc,
        "test_precision": test_prec,
        "test_recall": test_rec,
        "test_f1": test_f1,
        "test_roc_auc": test_roc,
        "test_pr_auc": test_pr,
        "confusion_matrix": cm.tolist(),
        "model_obj": model
    }

# =========================
# 7. Candidate models
# =========================
models = {
    "LogReg_balanced": Pipeline([
        ("pre", pre_onehot),
        ("clf", LogisticRegression(
            max_iter=2000,
            class_weight="balanced"
        ))
    ]),

    "SMOTE_LogReg": ImbPipeline([
        ("pre", pre_onehot),
        ("smote", SMOTE(random_state=42)),
        ("clf", LogisticRegression(
            max_iter=2000
        ))
    ]),

    "RF_balanced": Pipeline([
        ("pre", pre_onehot),
        ("clf", RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
            class_weight="balanced_subsample"
        ))
    ]),

    "XGB_weighted": Pipeline([
        ("pre", pre_ordinal),
        ("clf", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42,
            n_estimators=200,
            max_depth=4,
            learning_rate=0.08,
            subsample=0.9,
            colsample_bytree=0.9,
            min_child_weight=3,
            reg_lambda=2,
            scale_pos_weight=scale_pos_weight_value,
            n_jobs=-1
        ))
    ]),

    "XGB_weighted_stronger": Pipeline([
        ("pre", pre_ordinal),
        ("clf", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42,
            n_estimators=250,
            max_depth=4,
            learning_rate=0.06,
            subsample=0.9,
            colsample_bytree=0.9,
            min_child_weight=3,
            reg_lambda=2,
            scale_pos_weight=scale_pos_weight_value * 1.5,
            n_jobs=-1
        ))
    ]),
}

# =========================
# 8. Run all models
# =========================
results = []

for name, model in models.items():
    res = evaluate_model(name, model)
    results.append(res)

# =========================
# 9. Compare results
# =========================
results_df = pd.DataFrame([{
    "model": r["model"],
    "threshold": r["threshold"],
    "val_recall": r["val_recall"],
    "val_precision": r["val_precision"],
    "val_f1": r["val_f1"],
    "test_accuracy": r["test_accuracy"],
    "test_precision": r["test_precision"],
    "test_recall": r["test_recall"],
    "test_f1": r["test_f1"],
    "test_roc_auc": r["test_roc_auc"],
    "test_pr_auc": r["test_pr_auc"]
} for r in results])

results_df = results_df.sort_values(
    by=["test_recall", "test_f1", "test_precision"],
    ascending=False
).reset_index(drop=True)

print("\n==============================")
print("FINAL COMPARISON")
print("==============================")
print(results_df)

best_result = results_df.iloc[0]
print("\nBest model by recall/f1:")
print(best_result)

# =========================
# 10. Feature importance for best XGB if applicable
# =========================
best_model_name = best_result["model"]

if "XGB" in best_model_name:
    chosen = None
    for r in results:
        if r["model"] == best_model_name:
            chosen = r["model_obj"]
            break

    xgb_clf = chosen.named_steps["clf"]
    importances = xgb_clf.feature_importances_

    feature_names = ["physical_activity_level"] + num_cols
    imp_df = pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    }).sort_values("importance", ascending=False)

    print("\nFeature importance:")
    print(imp_df)

    plt.figure(figsize=(8, 5))
    plt.barh(imp_df["feature"], imp_df["importance"])
    plt.gca().invert_yaxis()
    plt.title(f"Feature Importance - {best_model_name}")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()

Class distribution:
has_disease
0    9851
1     149
Name: count, dtype: int64
Positive ratio: 0.0149

Shapes:
Train: (7000, 6)
Validation: (1500, 6)
Test: (1500, 6)

scale_pos_weight: 66.3076923076923

Running: LogReg_balanced
Best validation threshold: 0.15000000000000002
Validation recall: 1.0
Validation precision: 0.033093525179856115
Validation f1: 0.06406685236768803

Test Accuracy: 0.53
Test Precision: 0.028965517241379312
Test Recall: 0.9545454545454546
Test F1: 0.05622489959839357
Test ROC-AUC: 0.9066920900479764
Test PR-AUC: 0.26890107565760457

Test Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.52      0.69      1478
           1       0.03      0.95      0.06        22

    accuracy                           0.53      1500
   macro avg       0.51      0.74      0.37      1500
weighted avg       0.98      0.53      0.68      1500

Confusion Matrix:
[[774 704]
 [  1  21]]

Running: SMOTE_LogReg
Best validation thre